# Explicabilidade do Modelo e Interpretabilidade (SHAP)

Este notebook documenta a **Fase 6: Interpretação dos Resultados** do Tech Challenge.
Utilizamos SHAP (SHapley Additive exPlanations) para identificar quais variáveis
têm maior influência na classificação de qualidade do vinho e discutir as
implicações práticas para o processo de produção.

## 1. Carregamento do Modelo e Reconstrução dos Dados
Carregamos o modelo vencedor serializado e reconstruímos os dados de teste
com os mesmos splits determinísticos usados no treinamento.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import joblib
import numpy as np
import shap

from src.data_loader import load_wine_data
from src.preprocessing import create_target_variable, split_data
from src.features import engineer_features

# Carregar modelo
pipeline = joblib.load("../results/models/best_model.pkl")
print("Modelo carregado com sucesso.")

# Reconstruir dados
df_raw = load_wine_data()
df_target = create_target_variable(df_raw)
df_engineered = engineer_features(df_target)
id_cols = [c for c in ["Id", "id", "ID"] if c in df_engineered.columns]
X_train, X_test, y_train, y_test = split_data(
    df_engineered, target_col="high_quality", drop_cols=id_cols,
    test_size=0.2, random_state=42,
)
feature_names = list(X_test.columns)
print(f"Dados reconstruidos: {X_test.shape[0]} amostras, {len(feature_names)} features.")

## 2. Cálculo dos SHAP Values
O `TreeExplainer` é otimizado para modelos baseados em árvore e calcula
a contribuição exata de cada variável para cada predição individual.

In [ ]:
from src.interpretability import compute_shap_values

shap_values, explainer, X_processed = compute_shap_values(pipeline, X_test)
print(f"SHAP values calculados. Shape: {shap_values.shape}")

## 3. SHAP Summary Plot (Beeswarm)
Este gráfico mostra o impacto de cada variável nas predições.
Cada ponto é uma amostra. A cor indica o valor da feature (vermelho = alto, azul = baixo).
A posição horizontal indica o impacto na predição.

In [ ]:
shap.summary_plot(shap_values, X_processed, feature_names=feature_names)

## 4. Ranking de Importância (SHAP Bar)
Importância média absoluta de cada variável. Quanto maior, mais o modelo
depende dessa variável para tomar decisões.

In [ ]:
shap.plots.bar(shap.Explanation(values=shap_values, feature_names=feature_names))

## 5. Gráficos de Dependência (Top 3 Features)
Mostram como o valor de cada variável afeta a predição,
revelando relações não-lineares.

In [ ]:
for feat in ["sulphates", "alcohol", "volatile acidity"]:
    if feat in feature_names:
        shap.dependence_plot(feature_names.index(feat), shap_values, X_processed, feature_names=feature_names)

## 6. Interpretação e Implicações para Produção

### Top 5 Variáveis Mais Influentes:

1. **Sulfatos**: Conservantes naturais que protegem aroma e cor. Níveis adequados favorecem a qualidade.
2. **Razão Álcool/Densidade**: Captura corpo e leveza simultaneamente. Vinhos com boa estrutura alcoólica e baixa densidade são favorecidos.
3. **Acidez Volátil**: Impacto negativo. Níveis altos indicam fermentação problemática.
4. **Teor Alcoólico**: Reflete maturação da uva e fermentação completa.
5. **Razão de Enxofre**: Eficiência da proteção antioxidante.

### Recomendações para a Vinícola:
- Monitorar e otimizar sulfatos e teor alcoólico na fermentação.
- Implementar controles rigorosos contra acidez volátil.
- Utilizar o modelo como triagem automática de lotes antes da degustação.